# Phase 6 — Feature Engineering

This notebook converts the validated Phase 4 analytical layer into reusable match, innings, team-season, batting, and bowling summaries. It reads only `matches_clean.csv` and `deliveries_clean.csv`; raw and cleaned source files are never modified.

## 1. Load cleaned data

Paths are derived from the project location so the workflow remains portable.

In [1]:
from pathlib import Path
import hashlib
import sys

import pandas as pd
from IPython.display import display

candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / 'src').exists() else candidate.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_data_paths, load_clean_data, load_feature_data
from src.feature_engineering import (
    BOWLER_WICKET_TYPES,
    build_batting_summary,
    build_bowling_summary,
    build_innings_summary,
    build_match_features,
    build_team_season_summary,
    save_feature_datasets,
    validate_feature_datasets,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

paths = get_data_paths(PROJECT_ROOT)
source_keys = ['matches_raw', 'deliveries_raw', 'matches_clean', 'deliveries_clean']
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
source_hashes_before = {key: sha256(paths[key]) for key in source_keys}
matches, deliveries = load_clean_data(PROJECT_ROOT)
print(f'Matches: {matches.shape[0]:,} rows x {matches.shape[1]} columns')
print(f'Deliveries: {deliveries.shape[0]:,} rows x {deliveries.shape[1]} columns')

Matches: 1,095 rows x 32 columns
Deliveries: 260,920 rows x 21 columns


## 2. Confirm source schemas

The exact cleaned schemas are displayed before any aggregation. CSV date text is restored to a Pandas datetime by the loader.

In [2]:
display(pd.DataFrame({'matches_column': matches.columns, 'dtype': matches.dtypes.astype(str).values}))
display(pd.DataFrame({'deliveries_column': deliveries.columns, 'dtype': deliveries.dtypes.astype(str).values}))

,matches_column,dtype
0,match_id,int64
1,season_raw,str
2,season_standard,str
3,season_start_year,int64
4,date_raw,str
5,match_date,datetime64[us]
6,match_year,int64
7,match_month,int64
8,match_day,int64
9,match_type,str


,deliveries_column,dtype
0,match_id,int64
1,inning,int64
2,batting_team_raw,str
3,batting_team_standard,str
4,bowling_team_raw,str
5,bowling_team_standard,str
6,over,int64
7,ball,int64
8,batter,str
9,bowler,str


## 3. Feature definitions

The principal formulas are transparent and deliberately descriptive:

- Run rate = total runs / legal balls × 6.
- Win percentage = wins / (wins + losses) × 100; ties and no-results are excluded.
- Strike rate = batter runs / balls faced × 100; wides are not balls faced.
- Economy rate = bowler runs conceded / legal balls × 6; byes, leg-byes, and penalty runs are excluded from the bowler.
- Bowler wickets include bowled, caught, caught and bowled, lbw, stumped, and hit wicket.
- Dot balls are legal deliveries with zero total runs.

Rates remain null when their denominator is zero. Full definitions and limitations are in `docs/feature_definitions.md`.

## 4. Match/innings aggregation

One row is produced for each observed match and innings. Recorded, legal, and illegal deliveries stay distinct, and super-over innings remain separate.

In [3]:
innings_summary = build_innings_summary(matches, deliveries)
print(f'Innings summary: {len(innings_summary):,} rows')
display(innings_summary.head())

Innings summary: 2,217 rows


,match_id,season,inning,batting_team,bowling_team,recorded_deliveries,legal_balls,illegal_deliveries,total_runs,batsman_runs,extra_runs,wickets,bowler_wickets,fours,sixes,dot_balls,run_rate
0,335982,2007/08,1,Kolkata Knight Riders,Royal Challengers Bengaluru,124,120,4,222,205,17,3,3,15,14,36,11.10
1,335982,2007/08,2,Royal Challengers Bengaluru,Kolkata Knight Riders,101,91,10,82,63,19,10,9,3,3,50,5.41
2,335983,2007/08,1,Chennai Super Kings,Punjab Kings,124,120,4,240,234,6,5,5,20,16,34,12.00
3,335983,2007/08,2,Punjab Kings,Chennai Super Kings,124,120,4,207,196,11,4,4,18,9,23,10.35
4,335984,2007/08,1,Rajasthan Royals,Delhi Capitals,122,120,2,129,122,7,8,6,14,3,54,6.45


## 5. Match-level features

Outcome fields preserve source semantics: no-result matches receive neither a winner nor loser, and tied matches do not receive a manufactured losing team.

In [4]:
match_summary = build_match_features(matches, innings_summary)
print(f'Match summary: {len(match_summary):,} rows')
display(match_summary.head())

Match summary: 1,095 rows


,match_id,season_label,match_date,city,venue,team1,team2,toss_winner,toss_decision,winning_team,result,result_margin,player_of_match,match_type,super_over,method,match_decided,toss_winner_is_match_winner,losing_team,win_margin_type,match_stage,innings_count,match_recorded_deliveries,match_legal_balls,match_illegal_deliveries,match_total_runs,match_batsman_runs,match_extra_runs,match_wickets,match_bowler_wickets,match_fours,match_sixes,match_dot_balls
0,335982,2007/08,2008-04-18,Bengaluru,M Chinnaswamy Stadium,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,Kolkata Knight Riders,runs,140.00,BB McCullum,League,N,NaN,True,False,Royal Challengers Bengaluru,runs,League,2,225,211,14,304,268,36,13,12,18,17,86
1,335983,2007/08,2008-04-19,Chandigarh,Punjab Cricket Association IS Bindra Stadium,Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.00,MEK Hussey,League,N,NaN,True,True,Punjab Kings,runs,League,2,248,240,8,447,430,17,9,9,38,25,57
2,335984,2007/08,2008-04-19,Delhi,Feroz Shah Kotla,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,Delhi Capitals,wickets,9.00,MF Maharoof,League,N,NaN,True,False,Rajasthan Royals,wickets,League,2,219,211,8,261,244,17,9,7,32,4,85
3,335985,2007/08,2008-04-20,Mumbai,Wankhede Stadium,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,Royal Challengers Bengaluru,wickets,5.00,MV Boucher,League,N,NaN,True,False,Mumbai Indians,wickets,League,2,246,238,8,331,315,16,12,10,33,11,81
4,335986,2007/08,2008-04-20,Kolkata,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.00,DJ Hussey,League,N,NaN,True,False,Deccan Chargers,wickets,League,2,240,226,14,222,184,38,15,14,11,10,123


## 6. Team-season aggregation

Participation is counted from both match teams. Losses include only ordinary decided outcomes; ties and no-results have their own columns. Team scoring and conceding totals include every recorded innings, including super overs.

In [5]:
team_season_summary = build_team_season_summary(matches, deliveries)
print(f'Team-season summary: {len(team_season_summary):,} rows')
display(team_season_summary.head())

Team-season summary: 146 rows


,season,team,matches_played,decided_matches,wins,losses,ties,no_results,win_percentage,total_runs_scored,total_runs_conceded,average_runs_scored,average_runs_conceded,total_wickets_taken,total_wickets_lost,toss_wins,toss_decided_matches,toss_match_wins,toss_to_match_win_percentage
0,2007/08,Royal Challengers Bengaluru,14,14,4,10,0,0,28.57,1983,2205,141.64,157.50,56,89,5,5,1,20.00
1,2007/08,Punjab Kings,15,15,10,5,0,0,66.67,2464,2417,164.27,161.13,83,70,8,8,4,50.00
2,2007/08,Delhi Capitals,14,14,7,7,0,0,50.00,2118,2223,151.29,158.79,82,67,6,6,2,33.33
3,2007/08,Mumbai Indians,14,14,7,7,0,0,50.00,2080,2096,148.57,149.71,83,70,8,8,4,50.00
4,2007/08,Kolkata Knight Riders,13,13,6,7,0,0,46.15,1942,1718,149.38,132.15,61,77,6,6,3,50.00


## 7. Batting aggregation

Batting metrics use one row per season and batter. A ball faced excludes wides, dismissals are counted from supported dismissal records rather than appearances, and batting average remains null for a batter with no dismissal.

In [6]:
batting_summary = build_batting_summary(matches, deliveries)
print(f'Batting summary: {len(batting_summary):,} rows')
display(batting_summary.sort_values('runs', ascending=False).head())

Batting summary: 2,617 rows

,season,batter,matches,runs,balls_faced,fours,sixes,boundary_runs,dot_balls,dismissals,strike_rate,batting_average,boundary_percentage,dot_ball_percentage
1294,2016,V Kohli,16,973,640,84,38,564,165,12,152.03,81.08,57.97,25.78
2279,2023,Shubman Gill,17,890,564,85,33,538,144,15,157.80,59.33,60.45,25.53
2149,2022,JC Buttler,17,863,579,84,45,606,224,15,149.05,57.53,70.22,38.69
1299,2016,DA Warner,17,848,560,88,31,538,177,14,151.43,60.57,63.44,31.61
2446,2024,V Kohli,15,741,479,62,38,476,141,12,154.70,61.75,64.24,29.44


## 8. Bowling aggregation

Bowling metrics use the validated legal-delivery flag. Bowler wickets follow the documented convention and exclude run outs and other non-bowler dismissals.

In [7]:
bowling_summary = build_bowling_summary(matches, deliveries)
print(f'Bowling summary: {len(bowling_summary):,} rows')
print('Bowler-credit dismissal types:', ', '.join(sorted(BOWLER_WICKET_TYPES)))
display(bowling_summary.sort_values('wickets', ascending=False).head())

Bowling summary: 1,948 rows


Bowler-credit dismissal types: bowled, caught, caught and bowled, hit wicket, lbw, stumped


,season,bowler,matches,legal_balls,overs_bowled,runs_conceded,wickets,economy_rate,dot_balls,dot_ball_percentage,wides,no_balls
1442,2021,HV Patel,15,338,56.33,459,32,8.15,120,35.50,16,7
1357,2020/21,K Rabada,17,398,66.33,549,32,8.28,151,37.94,15,1
639,2013,DJ Bravo,18,375,62.50,497,32,7.95,135,36.00,16,1
1347,2020/21,JJ Bumrah,15,372,62.00,420,29,6.77,168,45.16,11,1
665,2013,JP Faulkner,16,379,63.17,427,28,6.76,161,42.48,16,0


## 9. Supporting Tableau metrics

The core outputs already contain boundary percentage, batting and bowling dot-ball percentages, match appearances, toss outcomes, and player-of-the-match values. Player-of-the-match counts can be aggregated from `match_summary` without another redundant file; a universal player-team association is not asserted because the source has no stable player master or player ID.

In [8]:
support_fields = {
    'match_summary': ['player_of_match', 'toss_winner_is_match_winner'],
    'batting_summary': ['matches', 'boundary_percentage', 'dot_ball_percentage'],
    'bowling_summary': ['matches', 'dot_ball_percentage'],
}
display(pd.DataFrame([(name, ', '.join(columns)) for name, columns in support_fields.items()], columns=['dataset', 'supporting_fields']))

,dataset,supporting_fields
0,match_summary,"player_of_match, toss_winner_is_match_winner"
1,batting_summary,"matches, boundary_percentage, dot_ball_percentage"
2,bowling_summary,"matches, dot_ball_percentage"


## 10. Data-quality validation

The shared validation routine checks grain uniqueness, valid domains and ranges, outcome semantics, formulas, and aggregate reconciliation to the cleaned delivery source. Saving is blocked if any check fails.

In [9]:
datasets = {
    'match_summary': match_summary,
    'innings_summary': innings_summary,
    'team_season_summary': team_season_summary,
    'batting_summary': batting_summary,
    'bowling_summary': bowling_summary,
}
validation = validate_feature_datasets(matches, deliveries, datasets)
display(validation)
assert validation['status'].eq('PASS').all(), validation.loc[validation['status'].ne('PASS')]
print(f"Validation: {validation['status'].eq('PASS').sum()}/{len(validation)} PASS")

,check,status,issue_count
0,Match summary has one row per source match,PASS,0
1,Innings summary grain is unique,PASS,0
2,Team-season grain is unique,PASS,0
3,Batting grain is unique,PASS,0
4,Bowling grain is unique,PASS,0
5,Match summary required dimensions are non-null,PASS,0
6,Innings summary required dimensions are non-null,PASS,0
7,Team-season summary required dimensions are no...,PASS,0
8,Batting summary required dimensions are non-null,PASS,0
9,Bowling summary required dimensions are non-null,PASS,0


Validation: 51/51 PASS


## 11. Reconciliation results

The headline totals below provide an auditable bridge from delivery-level sources to the analytical outputs.

In [10]:
reconciliation = pd.DataFrame({
    'metric': ['Total runs', 'Batter runs', 'Extras', 'Legal balls', 'Bowler-credit wickets'],
    'source_total': [deliveries.total_runs.sum(), deliveries.batsman_runs.sum(), deliveries.extra_runs.sum(), deliveries.legal_ball.sum(), deliveries.dismissal_kind.isin(BOWLER_WICKET_TYPES).sum()],
    'feature_total': [innings_summary.total_runs.sum(), batting_summary.runs.sum(), innings_summary.extra_runs.sum(), bowling_summary.legal_balls.sum(), bowling_summary.wickets.sum()],
})
reconciliation['difference'] = reconciliation['feature_total'] - reconciliation['source_total']
display(reconciliation)
assert reconciliation['difference'].eq(0).all()

,metric,source_total,feature_total,difference
0,Total runs,347756,347756,0
1,Batter runs,330064,330064,0
2,Extras,17692,17692,0
3,Legal balls,251471,251471,0
4,Bowler-credit wickets,11815,11815,0


## 12. Save and read back outputs

Only the five documented Phase 6 output files are written. Each is read back and checked against its in-memory shape and schema.

In [11]:
saved_paths = save_feature_datasets(datasets, PROJECT_ROOT)
readback = load_feature_data(PROJECT_ROOT)
for name, original in datasets.items():
    assert readback[name].shape == original.shape
    assert list(readback[name].columns) == list(original.columns)
output_profile = pd.DataFrame([
    {'dataset': name, 'rows': len(frame), 'columns': len(frame.columns), 'duplicate_key_rows': int(frame.duplicated({
        'match_summary': ['match_id'],
        'innings_summary': ['match_id', 'inning'],
        'team_season_summary': ['season', 'team'],
        'batting_summary': ['season', 'batter'],
        'bowling_summary': ['season', 'bowler'],
    }[name]).sum())}
    for name, frame in readback.items()
])
display(output_profile)

,dataset,rows,columns,duplicate_key_rows
0,match_summary,1095,33,0
1,innings_summary,2217,17,0
2,team_season_summary,146,19,0
3,batting_summary,2617,14,0
4,bowling_summary,1948,12,0


## 13. Source integrity

SHA-256 hashes taken before and after feature generation confirm that neither raw nor cleaned source datasets changed.

In [12]:
source_hashes_after = {key: sha256(paths[key]) for key in source_keys}
integrity = pd.DataFrame({
    'source': source_keys,
    'unchanged': [source_hashes_before[key] == source_hashes_after[key] for key in source_keys],
})
display(integrity)
assert integrity['unchanged'].all()

,source,unchanged
0,matches_raw,True
1,deliveries_raw,True
2,matches_clean,True
3,deliveries_clean,True


## 14. Phase 6 summary

Feature engineering is complete when all validation and reconciliation checks pass, every output reads back with its expected schema, and all four source hashes remain unchanged. The next project phase is **Phase 7 — SQL Analysis**; it is intentionally not started here.

In [13]:
print('Phase 6 feature engineering completed successfully.')
print('Generated:', ', '.join(path.name for path in saved_paths.values()))
print('Next phase: Phase 7 — SQL Analysis (not started).')

Phase 6 feature engineering completed successfully.
Generated: match_summary.csv, innings_summary.csv, team_season_summary.csv, batting_summary.csv, bowling_summary.csv
Next phase: Phase 7 — SQL Analysis (not started).
